# Vector Search and Embeddings

## Sentense Transformer 
Usage of sentense Transformer and dot (cosine similarity)

In [32]:
# Sentense Transformer - https://sbert.net/
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [33]:
q1 = "I just discovered the course, can I still join?"
v1 = model.encode(q1)
# v1.shape #       (384,) 
v1

array([-7.94798974e-03, -9.18932706e-02, -1.14074042e-02,  2.18466781e-02,
        1.11858100e-02, -1.30818253e-02, -7.39962459e-02, -9.87467021e-02,
       -1.05911456e-01, -3.03381626e-02, -2.92174835e-02,  2.33883113e-02,
        7.87485391e-03,  4.15800735e-02,  2.42720451e-02, -3.65723558e-02,
       -5.31095341e-02, -1.94869246e-02, -2.26979423e-02,  8.76776315e-03,
       -1.10785306e-01,  3.97906974e-02, -4.18479480e-02,  2.82960758e-02,
       -1.28302593e-02,  1.72567125e-02,  3.10428720e-02,  9.51102972e-02,
       -1.90717820e-02, -6.23235889e-02, -3.59100997e-02,  6.46180660e-02,
        3.06465216e-02,  2.39971988e-02,  2.06127595e-02,  9.87382606e-03,
        7.63836130e-02, -6.50511011e-02,  4.07934375e-03,  2.34528016e-02,
       -2.49406863e-02, -2.95020845e-02, -1.74878724e-02,  4.62139398e-02,
        2.48922762e-02,  1.08981781e-01, -5.67837805e-02, -6.95324615e-02,
        4.35407180e-03,  2.85132062e-02,  2.75795609e-02, -2.49844026e-02,
       -6.82143774e-03,  

In [34]:
q2 = "I just found out about the program, can I still enroll?"
v2 = model.encode(q2) 
v2.shape      # (384,)
#v2

(384,)

In [ ]:
import numpy as np

# Question has similar answer
q1 = "I just discovered the LLM course, can I still join?"
v1 = model.encode(q1)

res = "You can still join the LLM course."
res_v1 = model.encode(res)

dv = v1.dot(res_v1)
dv      # np.float32(0.9101957)  = cos(24)  = 24 angle means they are similar
print("dot value for q1 and res = ", dv)

angle_rad = np.arccos(dv)      
angle_deg = np.degrees(angle_rad) # 34.587418
print("cosine value for q1 and res = ", angle_deg)


# Question has unrelated answer
res2 = "Kubernetes orchestatres several docker containers effectively."
res_v2 = model.encode(res2)
dv2 = v1.dot(res_v2)
dv2     # np.float32(-0.13823347)

angle_rad = np.arccos(dv2)      # 1.7094738
angle_deg = np.degrees(angle_rad) # 97.945632
print("cosine of res2 = ", angle_deg)



dot value for q1 and res =  0.9101957
cosine value for q1 and res =  24.467587
cosine of res2 =  96.47518


## Embed FAQ dataset

In [71]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py



7[Files: 0  Bytes: 0  [0 B/s] Re]87[https://raw.githubusercontent.]87Saving 'ingest.py'
87ingest.py            100% [=============================>]     340     --.-KB/s87HTTP response 200  [https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py]
87ingest.py            100% [=============================>]     340     --.-KB/s87[Files: 1  Bytes: 340  [1.44KB/]8

In [ ]:
# Load FAQ JSON data

from ingest import load_faq_data
documents = load_faq_data()
len(documents)      # 1342
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [ ]:
# Filter data by taking question and answer only
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

len(texts)

1342

In [87]:
# Generate Embeddings with batch size 50
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1342

In [88]:
print("Each vector size = ", vectors[10].shape, v1.shape)       # (384,)

q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

v1.dot(vectors[10])


Each vector size =  (384,) (384,)


np.float32(0.33153284)

In [103]:
# Calculate Similarity of question and document
# Calculate the dot of each document stored in vectors array

scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

scores     # List of size 1342, each vector is size of 384

[np.float32(0.48740578),
 np.float32(0.2698118),
 np.float32(0.762941),
 np.float32(0.44378537),
 np.float32(0.26084003),
 np.float32(0.48665154),
 np.float32(0.3006155),
 np.float32(0.5601),
 np.float32(0.4596049),
 np.float32(0.2562817),
 np.float32(0.33153284),
 np.float32(0.27318525),
 np.float32(0.2769164),
 np.float32(0.34122998),
 np.float32(0.46007174),
 np.float32(0.26240283),
 np.float32(0.3920008),
 np.float32(0.10854163),
 np.float32(0.2756731),
 np.float32(0.16646823),
 np.float32(0.31437927),
 np.float32(0.006685349),
 np.float32(0.112050265),
 np.float32(0.21905695),
 np.float32(0.3400085),
 np.float32(0.23571289),
 np.float32(0.15088351),
 np.float32(0.16563259),
 np.float32(0.055450343),
 np.float32(0.23541187),
 np.float32(0.08533025),
 np.float32(-0.00308986),
 np.float32(-0.04259842),
 np.float32(-0.060277067),
 np.float32(0.006491054),
 np.float32(0.034277443),
 np.float32(-0.049589746),
 np.float32(-0.0006708056),
 np.float32(-0.017013557),
 np.float32(0.070627116

In [107]:
arr = np.array([[1,11], [2,22], [3,33], [4,44]])
arr.shape

(4, 2)

In [126]:
import numpy as np
X = np.array(vectors)
X        # X.shape returns (1342, 384) 

array([[-0.02670614, -0.12245762,  0.01594419, ..., -0.00230649,
        -0.112184  , -0.02365561],
       [-0.04383266, -0.09589145, -0.01538169, ...,  0.1184217 ,
        -0.00699406,  0.01887374],
       [-0.08896553, -0.06128182,  0.00775607, ...,  0.04059708,
         0.00479281, -0.02745941],
       ...,
       [-0.03652923,  0.01415431, -0.06838644, ...,  0.04316788,
         0.08105527, -0.0214863 ],
       [-0.1309159 , -0.06990605, -0.00931882, ..., -0.00044338,
        -0.01285736,  0.01426913],
       [-0.07984781,  0.01926989,  0.02544983, ..., -0.03368023,
        -0.01884022,  0.05837051]], shape=(1342, 384), dtype=float32)

In [127]:
scores2 = X.dot(v1)
scores2

array([ 0.4874058 ,  0.2698118 ,  0.762941  , ..., -0.08637968,
        0.03759799, -0.03037045], shape=(1342,), dtype=float32)

In [128]:
import json

idx = np.argmax(scores2)        # returns the index which has max value
print(idx, scores[idx])
print(q1, "\n\n", json.dumps(documents[idx], indent=2))
#print(q1, "\n\n", documents[idx]["answer"])

2 0.762941
Can I still join the course after the start date? 

 {
  "id": "3f1424af17",
  "course": "data-engineering-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "Course: Can I still join the course after the start date?",
  "answer": "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."
}


In [ ]:
top5 = np.argsort(scores2)[-5:]     # sort by ascending order, and take last 5
top5 = top5[::-1]                   # reverse order so that top 1 is first

print("Top 5 indices = ", top5)

# Print 
for idx in top5:
    print(scores2[idx])
    print(documents[idx])
    print()


Top 5 indices =  [  2 617 899 536   7]
0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579371
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.71921325
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomca